In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import root_mean_squared_error

from utils.data_splitter import DataSplitter
from utils.wrangle_model_data import wrangle_ml

df = pd.read_csv('data/commodity_prices.csv')
df = wrangle_ml(df)

date_dict = {
    'train_start': "2023-06-01", 'train_end': "2025-06-30",
    'valid_start': "2025-07-01", 'valid_end': "2025-07-31",
    'test_start': "2025-08-01", 'test_end': "2025-08-18"
}

thresholds = {
    'train': 250,
    'valid': 10,
    'test': 5
}

splitter = DataSplitter(df, date_dict, thresholds)
train_df, valid_df, test_df = splitter.run()

features = [col for col in train_df.columns if col not in ['log_Modal_Price', 'log_Modal_Price_filled', 'Arrival_Date']]
target_col = "log_Modal_Price_filled"

# Assuming you have your train and validation sets ready
X_train, y_train = train_df[features].copy(), train_df[target_col].copy()
X_val, y_val = valid_df[features].copy(), valid_df[target_col].copy()

categorical_cols = ['Product_Type', 'Commodity', 'Variety_Type', 'Market', 'Season', 'Market_Season', 'Variety_Type', 'Product_Month']

for c in categorical_cols:
    X_train[c] = X_train[c].astype('category')
    X_val[c] = X_val[c].astype('category')

best_params = {
    'learning_rate': 0.0439,       
    'num_leaves': 89,              
    'max_depth': 4,                 
    'min_child_samples': 64,       
    'subsample': 0.6932,           
    'colsample_bytree': 0.6071,     
    'reg_lambda': 6.13e-06        
}

# Create dataset objects for LightGBM
lgb_train = lgb.Dataset(X_train, y_train, categorical_feature=categorical_cols)
lgb_val = lgb.Dataset(X_val, y_val, reference=lgb_train, categorical_feature=categorical_cols)

# Train the model
model = lgb.train(
    best_params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[
                lgb.early_stopping(stopping_rounds=100),
                lgb.log_evaluation(period=0) 
            ]
)

# Predict on validation set
y_pred = model.predict(X_val, num_iteration=model.best_iteration)

# Create a mask for rows where the original price exists
mask_val = valid_df['log_Modal_Price'].notna()

# Filter validation target and predictions
y_val_actual = y_val[mask_val]
y_pred_actual = y_pred[mask_val]

# Compute RMSE
rmse = root_mean_squared_error(y_val_actual, y_pred_actual)

[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003767 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4664
[LightGBM] [Info] Number of data points in the train set: 204620, number of used features: 27
[LightGBM] [Info] Start training from score 8.364827
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
Training until validation scores don't improve for 100 rounds
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


In [2]:
print(f"Validation RMSE: {rmse:.5f}")

Validation RMSE: 0.11012
